# QaptaanLM-0.75B Supervised Fine-Tuning (SFT) Pipeline
### Kaggle TPU v5e-8 Acceleration Notebook (8 Pod Cores — Native JAX/Flax BF16)

This notebook runs Stage 2 full-parameter **Supervised Fine-Tuning (SFT)** on **QaptaanLM-0.75B** with the **KapInstruct-100M** dataset using Google TPU v5e-8 (8 TPU cores, 128 GB total HBM):

| Feature | TPU v5e-8 SFT Setup |
|:---|:---|
| **Base Model** | `kaptaan45/QaptaanLM-0.75B` (CPT Foundation Checkpoint) |
| **Dataset** | `kaptaan45/KapInstruct-100M` (100M tokens, ChatML schema) |
| **Loss Policy** | Assistant-Only Loss Masking (`labels = -100` for prompts) |
| **Compute Architecture** | 8 TPU v5e Cores (~1,576 TFLOPS BF16 Peak) |
| **Memory** | 128 GB Total HBM (16 GB per core) |
| **Precision** | Native Hardware `bfloat16` |
| **Learning Rate** | `5.0e-6` with full cosine decay to `0.0` |
| **Target Duration** | **~50–65 Minutes** (Single Kaggle session) |

**KapInstruct-100M Composition (12 High-Signal Domains):**
- 31% Code Generation (`Magicoder-Evol`, `Magicoder-OSS`, `StarCoder2-Exec`, `Smol-Constraints`)
- 27% General Dialogue & Reasoning (`Smol-Magpie-Ultra`, `OpenHermes-2.5`)
- 17% Mathematics CoT (`OpenMathInstruct-2`, `NuminaMath-CoT`)
- 11% STEM QA & Science (`OpenThoughts-114k`, `WebInstructSub`)
- 10% Debugging & Error Repair (`CodeFeedback-Filtered`)
- 4% Strict Constraint Adherence (`Tulu-3-SFT`, `Smol-Constraints`)

## 0. Directory Setup & Repository Working Path

In [ ]:
import os, sys
# Ensure working directory is set to QaptaanLM-0.75B
if not os.path.exists("/kaggle/working/QaptaanLM-0.75B"):
    !git clone https://github.com/rudy-07/QaptaanLM-0.75B.git /kaggle/working/QaptaanLM-0.75B

os.chdir("/kaggle/working/QaptaanLM-0.75B")
print(f"✓ Working directory: {os.getcwd()}")
sys.path.insert(0, os.getcwd())

## 1. Install JAX TPU & Training Dependencies

In [ ]:
# Install TPU-optimized JAX, Flax, Optax, Orbax, and Hugging Face libraries
!pip install -q "jax[tpu]" flax optax orbax-checkpoint -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
!pip install -q transformers>=5.13.0 datasets pyarrow safetensors huggingface_hub pyyaml rich

## 2. Verify TPU v5e-8 Hardware & JAX Devices

In [ ]:
import jax
devices = jax.devices()
print(f"✓ JAX version: {jax.__version__}")
print(f"✓ Device backend: {jax.default_backend()}")
print(f"✓ Total TPU devices ({len(devices)}): {[str(d) for d in devices]}")
assert len(devices) == 8, f"Expected 8 TPU devices on TPU v5e-8, found {len(devices)}"

## 3. Hugging Face Authentication

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✓ Logged in to Hugging Face")
except Exception as e:
    print(f"Manual login needed or secret not found: {e}")

## 4. Prepare KapInstruct-100M Dataset Shards

In [ ]:
import glob
os.makedirs("/kaggle/working/data", exist_ok=True)
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
os.makedirs("/kaggle/working/logs", exist_ok=True)

# Locate attached Kaggle dataset shards
input_arrow = glob.glob("/kaggle/input/**/*.arrow", recursive=True)
input_parquet = glob.glob("/kaggle/input/**/*.parquet", recursive=True)

if input_arrow or input_parquet:
    found = input_arrow or input_parquet
    print(f"✓ Found {len(found)} dataset shards in Kaggle input:")
    for f in found[:3]:
        print(f"  - {f}")
    data_path = os.path.dirname(found[0])
else:
    # Fallback to direct HF streaming / download
    print("Downloading KapInstruct-100M shards directly from Hugging Face...")
    from datasets import load_dataset
    ds = load_dataset("kaptaan45/KapInstruct-100M", split="train")
    ds.save_to_disk("/kaggle/working/data/kapinstruct")
    data_path = "/kaggle/working/data"

## 5. Run SFT Pipeline Smoke Test (5 Steps)

In [ ]:
# Quick smoke test to compile TPU computation graph and verify assistant-only loss masking
!python -m jax_training.train --mode sft --smoke-test

## 6. Launch Full Supervised Fine-Tuning (100M Tokens)

Runs full SFT training with in-training validation on held-out split and automatic Hugging Face safetensors export upon completion.

In [ ]:
# Execute full SFT training on TPU v5e-8
!python -m jax_training.train \
    --mode sft \
    --config configs/sft_config.yaml \
    --data-dir {data_path} \
    --export-hf

## 7. Interactive Inference & ChatML Dialogue Verification

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

export_dir = "checkpoints/jax_sft_hf"
print(f"Loading exported SFT model from {export_dir}...")

tokenizer = AutoTokenizer.from_pretrained(export_dir, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    export_dir,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

test_prompts = [
    "Write a Python function to compute the Levenshtein distance between two strings with full type annotations.",
    "Explain why quicksort has an average time complexity of O(n log n) but worst-case O(n^2).",
    "Write a SQL query to find the top 3 highest earning employees in each department from an Employee(id, name, salary, dept_id) table."
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": "You are QaptaanLM, an expert programming and reasoning assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt")
    
    print("=" * 70)
    print(f"PROMPT: {prompt}")
    print("=" * 70)
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.2,
            top_p=0.95,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    resp = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(resp)
    print()

## 8. Push SFT Model to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi

repo_id = "kaptaan45/QaptaanLM-0.75B"  # or 'kaptaan45/QaptaanLM-0.75B-Instruct'
print(f"Pushing fine-tuned SFT weights to Hugging Face: {repo_id}...")

api = HfApi()
api.upload_folder(
    folder_path="checkpoints/jax_sft_hf",
    repo_id=repo_id,
    repo_type="model",
    commit_message="Stage 2: Supervised Fine-Tuning (SFT) on KapInstruct-100M"
)
print(f"✓ Successfully uploaded model to https://huggingface.co/{repo_id}")